In [1]:
!pip install -q -U qdrant-client fastembed pandas openpyxl sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 103.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 73.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [3]:
%%writefile config.py
import os

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]

COLLECTION_NAME = "thu_tuc_hanh_chinh"

EXCEL_PATH = "/kaggle/input/datasets/thngtrnnh/datatest/thu_tuc.xlsx"
DATA_EVAL_PATH = "/kaggle/input/datasets/thngtrnnh/datatest/evaluation_dataset_80_20.jsonl"
SHEET_NAME = 0

COL_MA_SO = "Mã số"
COL_TEN_THU_TUC = "Tên"
COL_CO_QUAN_BAN_HANH = "Cơ quan ban hành"
COL_CO_QUAN_THUC_HIEN = "Cơ quan thực hiện"
COL_LINH_VUC = "Lĩnh vực"

DENSE_MODEL_NAME = "intfloat/multilingual-e5-large"
DENSE_VECTOR_SIZE = 1024

SPARSE_MODEL_NAME = "Qdrant/bm25"


Writing config.py


In [4]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["QDRANT_URL"] = user_secrets.get_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = user_secrets.get_secret("QDRANT_API_KEY")


In [5]:
%%writefile ingest_gpu.py
import pandas as pd
import torch
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding,TextEmbedding
# from sentence_transformers import SentenceTransformer

import config as cfg


def main():
    print(f"Đang đọc file: {cfg.EXCEL_PATH}")
    df = pd.read_excel(cfg.EXCEL_PATH, sheet_name=cfg.SHEET_NAME)
    df.columns = df.columns.str.strip()

    missing_cols = [c for c in (cfg.COL_MA_SO, cfg.COL_TEN_THU_TUC) if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Không tìm thấy cột {missing_cols} trong file Excel. "
            f"Các cột hiện có: {list(df.columns)}"
        )

    payload_key_map = {
        cfg.COL_CO_QUAN_BAN_HANH: "co_quan_ban_hanh",
        cfg.COL_CO_QUAN_THUC_HIEN: "co_quan_thuc_hien",
        cfg.COL_LINH_VUC: "linh_vuc",
    }
    available_meta_cols = [c for c in payload_key_map if c in df.columns]

    df = df.dropna(subset=[cfg.COL_TEN_THU_TUC]).reset_index(drop=True)
    print(f"Đọc được {len(df)} dòng dữ liệu.")

    client = QdrantClient(url=cfg.QDRANT_URL, api_key=cfg.QDRANT_API_KEY)

    if not client.collection_exists(cfg.COLLECTION_NAME):
        print(f"Tạo collection '{cfg.COLLECTION_NAME}'...")
        client.create_collection(
            collection_name=cfg.COLLECTION_NAME,
            vectors_config={
                "dense": models.VectorParams(
                    size=cfg.DENSE_VECTOR_SIZE,
                    distance=models.Distance.COSINE,
                ),
            },
            sparse_vectors_config={
                "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF),
            },
        )
    else:
        print(f"Collection '{cfg.COLLECTION_NAME}' đã tồn tại, sẽ upsert thêm/ghi đè dữ liệu.")

    for excel_col in available_meta_cols:
        payload_key = payload_key_map[excel_col]
        try:
            client.create_payload_index(
                collection_name=cfg.COLLECTION_NAME,
                field_name=payload_key,
                field_schema=models.PayloadSchemaType.KEYWORD,
            )
        except Exception:
            pass

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Dense device:", device)

    dense_model = TextEmbedding(cfg.DENSE_MODEL_NAME, device=device)
    sparse_model = SparseTextEmbedding(cfg.SPARSE_MODEL_NAME)

    texts = df[cfg.COL_TEN_THU_TUC].astype(str).tolist()

    print("Đang sinh dense embeddings bằng GPU...")
    dense_vecs = dense_model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    print("Đang sinh sparse BM25 embeddings...")
    sparse_vecs = list(sparse_model.embed(texts))

    BATCH_SIZE = 100
    points = []

    for i, row in df.iterrows():
        sv = sparse_vecs[i]
        payload = {
            "ma_so": str(row[cfg.COL_MA_SO]),
            "ten_thu_tuc": str(row[cfg.COL_TEN_THU_TUC]),
        }
        for excel_col, payload_key in payload_key_map.items():
            if excel_col in available_meta_cols and pd.notna(row[excel_col]):
                payload[payload_key] = str(row[excel_col]).strip()

        points.append(
            models.PointStruct(
                id=i,
                vector={
                    "dense": dense_vecs[i].tolist(),
                    "bm25": models.SparseVector(
                        indices=sv.indices.tolist(),
                        values=sv.values.tolist(),
                    ),
                },
                payload=payload,
            )
        )

    print(f"Đang upload {len(points)} điểm vào Qdrant...")
    for start in range(0, len(points), BATCH_SIZE):
        batch = points[start:start + BATCH_SIZE]
        client.upsert(collection_name=cfg.COLLECTION_NAME, points=batch, wait=True)
        print(f"  Đã upload {min(start + BATCH_SIZE, len(points))}/{len(points)}")

    print("Hoàn tất ingest!")


if __name__ == "__main__":
    main()


Writing ingest_gpu.py


In [34]:
!python ingest_gpu.py


Đang đọc file: /kaggle/input/datasets/thngtrnnh/datatest/thu_tuc.xlsx
Đọc được 5764 dòng dữ liệu.
Collection 'thu_tuc_hanh_chinh' đã tồn tại, sẽ upsert thêm/ghi đè dữ liệu.
Dense device: cuda
Loading weights: 100%|█| 391/391 [00:00<00:00, 1543.30it/s, Materializing param=
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Đang sinh dense embeddings bằng GPU...
Batches: 100%|██████████████████████████████████| 91/91 [00:52<00:00,  1.72it/s]
Đang sinh sparse BM25 embeddings...
Đang upload 5764 điểm vào Qdrant...
  Đã upload 100/5764
  Đã upload 200/5764
  Đã upload 300/5764
  Đã upload 400/5764
  Đã upload 500/5764
  Đã upload 600/5764
  Đã upload 700/5764
  Đã upload 800/5764
  Đã upload 900/5764
  Đã upload 1000/5764
  Đã uplo

In [15]:
%%writefile search_gpu.py
import sys
import torch
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding, TextEmbedding
# from sentence_transformers import SentenceTransformer

import config as cfg

client = QdrantClient(url=cfg.QDRANT_URL, api_key=cfg.QDRANT_API_KEY)

device = "cuda" if torch.cuda.is_available() else "cpu"
dense_model = TextEmbedding(cfg.DENSE_MODEL_NAME, device=device)
sparse_model = SparseTextEmbedding(cfg.SPARSE_MODEL_NAME)


def build_filter(co_quan_ban_hanh=None, co_quan_thuc_hien=None, linh_vuc=None):
    conditions = []
    if co_quan_ban_hanh:
        conditions.append(
            models.FieldCondition(key="co_quan_ban_hanh", match=models.MatchValue(value=co_quan_ban_hanh))
        )
    if co_quan_thuc_hien:
        conditions.append(
            models.FieldCondition(key="co_quan_thuc_hien", match=models.MatchValue(value=co_quan_thuc_hien))
        )
    if linh_vuc:
        conditions.append(
            models.FieldCondition(key="linh_vuc", match=models.MatchValue(value=linh_vuc))
        )
    return models.Filter(must=conditions) if conditions else None


def search(query: str, limit: int = 5, co_quan_ban_hanh=None, co_quan_thuc_hien=None, linh_vuc=None):
    dense_query = list(dense_model.embed([query]))[0]
    sparse_query = list(sparse_model.embed([query]))[0]
    query_filter = build_filter(co_quan_ban_hanh, co_quan_thuc_hien, linh_vuc)

    results = client.query_points(
        collection_name=cfg.COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=dense_query.tolist(), using="dense", limit=20, filter=query_filter),
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_query.indices.tolist(),
                    values=sparse_query.values.tolist(),
                ),
                using="bm25",
                limit=20,
                filter=query_filter,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
    )
    return results.points


if __name__ == "__main__":
    query = " ".join(sys.argv[1:]) or input("Nhập câu truy vấn: ")
    hits = search(query)

    print(f"\\nKết quả cho: '{query}'\\n" + "-" * 50)
    for i, hit in enumerate(hits, 1):
        p = hit.payload
        print(f"{i}. [{p['ma_so']}] {p['ten_thu_tuc']}  (score={hit.score:.4f})")


Overwriting search_gpu.py


In [16]:
%%writefile eval_gpu.py
import argparse
import json
from pathlib import Path
from typing import Dict, List

from search_gpu import search

DATASET_PATH = Path("/kaggle/input/datasets/thngtrnnh/datatest/evaluation_dataset_80_20.jsonl")

def load_evaluation_set(dataset_path: Path) -> List[Dict]:
    evaluation_set = []
    seen_query_ids = set()

    with dataset_path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            item = json.loads(line)
            required_keys = {"query_id", "query_text", "expected_doc_ids"}
            missing_keys = required_keys - set(item.keys())
            if missing_keys:
                raise ValueError(f"Dòng {line_no} thiếu các trường bắt buộc: {sorted(missing_keys)}")

            if item["query_id"] in seen_query_ids:
                raise ValueError(f"query_id bị trùng: {item['query_id']}")
            seen_query_ids.add(item["query_id"])

            if not isinstance(item["expected_doc_ids"], list) or not item["expected_doc_ids"]:
                raise ValueError(f"Dòng {line_no} phải có expected_doc_ids là list không rỗng")

            evaluation_set.append(item)

    if not evaluation_set:
        raise ValueError("Dataset rỗng.")
    return evaluation_set


def hybrid_search(query_text: str, top_k: int = 5) -> List[str]:
    hits = search(query_text, limit=top_k)
    retrieved_ids = []
    for hit in hits:
        ma_so = str(hit.payload.get("ma_so", "")).strip()
        if ma_so:
            retrieved_ids.append(ma_so)
    return retrieved_ids


def calculate_metrics(retrieved_ids: List[str], expected_ids: List[str]) -> Dict:
    expected_set = set(expected_ids)
    retrieved_set = set(retrieved_ids)

    num_relevant_retrieved = len(expected_set & retrieved_set)
    hit = 1 if num_relevant_retrieved > 0 else 0
    recall = num_relevant_retrieved / len(expected_set) if expected_set else 0.0
    precision = num_relevant_retrieved / len(retrieved_ids) if retrieved_ids else 0.0

    mrr = 0.0
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in expected_set:
            mrr = 1.0 / rank
            break

    return {
        "hit": hit,
        "recall": recall,
        "precision": precision,
        "mrr": mrr,
    }


def evaluate_retrieval(eval_set: List[Dict], top_k: int = 15, show_only_errors: bool = False):
    total_queries = len(eval_set)
    sums = {"hit": 0, "recall": 0.0, "precision": 0.0, "mrr": 0.0}
    error_count = 0

    for item in eval_set:
        retrieved_ids = hybrid_search(item["query_text"], top_k=top_k)
        expected_ids = item["expected_doc_ids"]
        metrics = calculate_metrics(retrieved_ids, expected_ids)

        for key in sums:
            sums[key] += metrics[key]

        is_error = metrics["hit"] == 0
        if is_error:
            error_count += 1

        if not show_only_errors or is_error:
            print(item["query_id"], expected_ids, retrieved_ids, metrics)
            print("  Query:", item["query_text"])

    print("\\nTổng số query:", total_queries)
    print("Số query trượt hoàn toàn (Hit=0):", error_count)
    print(f"Hit Rate     : {sums['hit'] / total_queries:.2%}")
    print(f"Recall@{top_k}    : {sums['recall'] / total_queries:.2%}")
    print(f"Precision@{top_k} : {sums['precision'] / total_queries:.2%}")
    print(f"MRR          : {sums['mrr'] / total_queries:.4f}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", default=str(DATASET_PATH))
    parser.add_argument("--top-k", type=int, default=15)
    parser.add_argument("--only-errors", action="store_true")
    args = parser.parse_args()

    dataset = load_evaluation_set(Path(args.dataset))
    evaluate_retrieval(dataset, top_k=args.top_k, show_only_errors=args.only_errors)


Overwriting eval_gpu.py


In [17]:
!python eval_gpu.py --top-k 15


/kaggle/working/search_gpu.py:12: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_model = TextEmbedding(cfg.DENSE_MODEL_NAME, device=device)
q001 ['1.001193'] ['1.001193', '1.008342', '1.008335', '1.004884', '1.001020', '1.012289', '1.014507', '1.004772', '2.002794', '1.000689', '1.014331', '1.014070', '1.014088', '1.014332', '1.011445'] {'hit': 1, 'recall': 1.0, 'precision': 0.06666666666666667, 'mrr': 1.0}
  Query: Bé nhà tôi mới sinh, giờ muốn làm giấy khai sinh thì bắt đầu từ đâu?
q002 ['1.001193'] ['1.001020', '1.003220', '1.001193', '2.002765', '1.000689', '2.000547', '1.004884', '1.001695', '1.004772', '1.000110', '1.003583', '2.002755', '2.002757', '2.000712', '2.001023'] {'hit': 1, 'recall': 1.0, 'precision': 0.06666666666666667, 'mrr': 0.3333333333333333}
  Query: Làm khai sinh 

In [40]:
from fastembed import TextEmbedding

models = TextEmbedding.list_supported_models()
for m in models:
    print(m["model"], "-", m["dim"])

BAAI/bge-base-en - 768
BAAI/bge-base-en-v1.5 - 768
BAAI/bge-large-en-v1.5 - 1024
BAAI/bge-small-en - 384
BAAI/bge-small-en-v1.5 - 384
BAAI/bge-small-zh-v1.5 - 512
mixedbread-ai/mxbai-embed-large-v1 - 1024
snowflake/snowflake-arctic-embed-xs - 384
snowflake/snowflake-arctic-embed-s - 384
snowflake/snowflake-arctic-embed-m - 768
snowflake/snowflake-arctic-embed-m-long - 768
snowflake/snowflake-arctic-embed-l - 1024
jinaai/jina-clip-v1 - 768
Qdrant/clip-ViT-B-32-text - 512
sentence-transformers/all-MiniLM-L6-v2 - 384
jinaai/jina-embeddings-v2-base-en - 768
jinaai/jina-embeddings-v2-small-en - 512
jinaai/jina-embeddings-v2-base-de - 768
jinaai/jina-embeddings-v2-base-code - 768
jinaai/jina-embeddings-v2-base-zh - 768
jinaai/jina-embeddings-v2-base-es - 768
thenlper/gte-base - 768
thenlper/gte-large - 1024
nomic-ai/nomic-embed-text-v1.5 - 768
nomic-ai/nomic-embed-text-v1.5-Q - 768
nomic-ai/nomic-embed-text-v1 - 768
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 - 384
sentence-t